# Les fichiers en Python
## Tutoriel complet : lecture, écriture et manipulation

---

## 1. Ouvrir et fermer un fichier

Pour travailler avec un fichier, on utilise la fonction `open()` qui prend deux paramètres :
- Le **chemin** du fichier
- Le **mode** d'ouverture

| Mode | Description | Si le fichier n'existe pas |
|---|---|---|
| `"r"` | Lecture seule (par défaut) | Lève `FileNotFoundError` |
| `"w"` | Écriture (écrase tout le contenu) | Crée le fichier |
| `"a"` | Ajout (écrit à la fin) | Crée le fichier |
| `"x"` | Création exclusive : échoue si le fichier existe déjà (`FileExistsError`) | Crée le fichier |
| `"r+"` | Lecture **et** écriture (curseur au début, ne vide pas le fichier) | Lève `FileNotFoundError` |
| `"w+"` | Écriture **et** lecture (vide le fichier à l'ouverture) | Crée le fichier |
| `"a+"` | Ajout **et** lecture (écrit toujours à la fin) | Crée le fichier |
| `"b"` | Binaire : à combiner avec un autre mode (`"rb"`, `"wb"`...) pour lire/écrire des octets bruts (images, PDF...) au lieu de texte | Selon le mode combiné |

**Attention** : le mode `"w"` **efface tout** le contenu du fichier dès l'ouverture.

In [1]:
# Méthode classique : ouvrir, travailler, fermer
fichier = open("test.txt", "w")
fichier.write("Bonjour depuis Python !\n")
fichier.close()   # TOUJOURS fermer le fichier après

print("Fichier créé et fermé.")

Fichier créé et fermé.


### Le problème du `close()` oublié

Si le programme plante avant le `close()`, le fichier reste ouvert et les données peuvent être perdues.

```python
# ❌ DANGEREUX
fichier = open("data.txt", "r")
contenu = fichier.read()
traitement(contenu)  # si ça plante ici, close() n'est jamais appelé
fichier.close()
```

---

## 2. La bonne pratique : `with` (context manager)

Le mot-clé `with` ferme **automatiquement** le fichier à la fin du bloc, même si une erreur survient.

**Règle : utilise toujours `with`.**

In [ ]:
# ✅ Bonne pratique : with ferme le fichier automatiquement
with open("test.txt", "w") as fichier:
    fichier.write("Ligne 1\n")
    fichier.write("Ligne 2\n")
    fichier.write("Ligne 3\n")

# Ici le fichier est déjà fermé, pas besoin de close()
print("Fichier écrit et fermé automatiquement.")

---

## 3. Écrire dans un fichier

### 3.1 — `write()` : écrire une chaîne

`write()` écrit du texte dans le fichier mais **n'ajoute pas** de retour à la ligne automatiquement. Tu dois mettre `\n` toi-même.

In [3]:
# write() n'ajoute PAS de \n automatiquement
with open("test.txt", "w") as fichier:
    fichier.write("Bonjour\n")      # ← \n à mettre soi-même
    fichier.write("Au revoir\n")

# Sans les \n, tout serait collé : "BonjourAu revoir"

# Vérification
with open("test.txt", "r") as fichier:
    print(fichier.read())

Bonjour
Au revoir



### 3.2 — `writelines()` : écrire une liste

Écrit une liste de chaînes d'un coup, mais **sans ajouter de `\n`** entre elles.

In [ ]:
# writelines() écrit une liste mais SANS ajouter de \n
noms = ["Alice", "Bob", "Charlie"]

with open("demo_writelines.txt", "w") as fichier:
    # On ajoute \n à chaque élément
    fichier.writelines(nom + "\n" for nom in noms)

# Vérification
with open("demo_writelines.txt", "r") as fichier:
    print(fichier.read())

### 3.3 — `print()` dans un fichier

On peut rediriger `print()` vers un fichier avec le paramètre `file=`. L'avantage : `print()` ajoute le `\n` automatiquement.

In [ ]:
# print() avec file= ajoute le \n automatiquement
with open("demo_print.txt", "w") as fichier:
    print("Bonjour", file=fichier)        # \n automatique
    print("Au revoir", file=fichier)
    print(42, True, file=fichier)          # convertit en string tout seul

# Vérification
with open("demo_print.txt", "r") as fichier:
    print(fichier.read())

### 3.4 — Mode ajout `"a"` : écrire à la fin

Le mode `"a"` (append) ajoute du contenu **à la fin** sans effacer ce qui existe déjà. Idéal pour les logs ou l'historique.

In [ ]:
# On crée le fichier avec "w"
with open("log.txt", "w") as fichier:
    fichier.write("Entrée 1\n")

# On ajoute à la fin avec "a" (sans effacer)
with open("log.txt", "a") as fichier:
    fichier.write("Entrée 2\n")
    fichier.write("Entrée 3\n")

# Vérification : les 3 entrées sont là
with open("log.txt", "r") as fichier:
    print(fichier.read())

### ⚠️ Différence entre `"w"` et `"a"`

In [ ]:
# Démonstration : "w" ÉCRASE tout à chaque ouverture
with open("ecrase.txt", "w") as fichier:
    fichier.write("Premier contenu\n")

with open("ecrase.txt", "w") as fichier:
    fichier.write("Deuxième contenu\n")   # ← le premier est perdu !

with open("ecrase.txt", "r") as fichier:
    print("Avec 'w' :")
    print(fichier.read())   # Affiche seulement "Deuxième contenu"

print("---")

# Démonstration : "a" AJOUTE à la fin
with open("ajoute.txt", "w") as fichier:
    fichier.write("Premier contenu\n")

with open("ajoute.txt", "a") as fichier:
    fichier.write("Deuxième contenu\n")   # ← le premier est gardé !

with open("ajoute.txt", "r") as fichier:
    print("Avec 'a' :")
    print(fichier.read())   # Affiche les deux lignes

---

## 4. Lire un fichier

On va d'abord créer un fichier de test pour les exemples suivants.

In [ ]:
# Création d'un fichier de test
with open("lecture_test.txt", "w") as fichier:
    fichier.write("Première ligne\n")
    fichier.write("Deuxième ligne\n")
    fichier.write("Troisième ligne\n")
    fichier.write("Quatrième ligne\n")
    fichier.write("Cinquième ligne\n")

print("Fichier de test créé.")

### 4.1 — `read()` : tout lire d'un coup

Lit **tout** le contenu du fichier dans une seule chaîne de caractères.

**Attention** : charge tout en mémoire. Pour un fichier de 2 Go, ça prendra 2 Go de RAM.

In [ ]:
# read() retourne TOUT le fichier dans une seule string
with open("lecture_test.txt", "r") as fichier:
    contenu = fichier.read()

print(contenu)
print(f"Type : {type(contenu)}")
print(f"Longueur : {len(contenu)} caractères")

In [ ]:
# On peut aussi lire un nombre limité de caractères
with open("lecture_test.txt", "r") as fichier:
    debut = fichier.read(20)   # seulement les 20 premiers caractères

print(f"Les 20 premiers caractères : '{debut}'")

### 4.2 — `readline()` : lire une seule ligne

Lit la ligne suivante à chaque appel. Le `\n` est **inclus** dans le résultat.

In [ ]:
# readline() lit UNE ligne à chaque appel
with open("lecture_test.txt", "r") as fichier:
    ligne1 = fichier.readline()   # "Première ligne\n"
    ligne2 = fichier.readline()   # "Deuxième ligne\n"

print(f"Ligne 1 brute : {repr(ligne1)}")      # repr() montre les \n
print(f"Ligne 1 propre : {ligne1.strip()}")    # strip() enlève le \n
print(f"Ligne 2 propre : {ligne2.strip()}")

### 4.3 — `readlines()` : toutes les lignes dans une liste

Retourne une **liste** où chaque élément est une ligne (avec le `\n` inclus).

In [ ]:
# readlines() retourne une LISTE de lignes
with open("lecture_test.txt", "r") as fichier:
    lignes = fichier.readlines()

print(f"Type : {type(lignes)}")
print(f"Nombre de lignes : {len(lignes)}")
print(f"Brut : {lignes}")
print()

# Pour nettoyer les \n :
lignes_propres = [ligne.strip() for ligne in lignes]
print(f"Propre : {lignes_propres}")

### 4.4 — La boucle `for` : la meilleure méthode ⭐

Parcourir le fichier ligne par ligne avec un `for` est la méthode **recommandée**. Elle est économe en mémoire car elle ne charge qu'une ligne à la fois.

In [ ]:
# ⭐ LA MEILLEURE MÉTHODE : boucle for
# Charge une seule ligne à la fois en mémoire
with open("lecture_test.txt", "r") as fichier:
    for numero, ligne in enumerate(fichier, 1):
        print(f"Ligne {numero} : {ligne.strip()}")

### Récapitulatif lecture

| Méthode | Retourne | Mémoire | Quand l'utiliser |
|---|---|---|---|
| `read()` | Une seule string | Charge tout | Petits fichiers |
| `readline()` | Une ligne (string) | Une ligne | Lire les premières lignes |
| `readlines()` | Liste de strings | Charge tout | Quand tu veux une liste |
| `for ligne in fichier` | Une ligne par tour | Une ligne | **Toujours (recommandé)** |

---

## 5. L'encodage

L'encodage détermine comment les caractères (accents, émojis...) sont stockés en mémoire.

| Encodage | Usage |
|---|---|
| `utf-8` | Standard universel (recommandé) |
| `latin-1` | Vieux fichiers européens |
| `cp1252` | Windows par défaut |
| `ascii` | Anglais uniquement (pas d'accents) |

**Règle : spécifie toujours `encoding="utf-8"`** pour éviter les problèmes d'accents.

In [ ]:
# ✅ Toujours préciser l'encodage
with open("accents.txt", "w", encoding="utf-8") as fichier:
    fichier.write("Café, crème brûlée, naïf\n")
    fichier.write("Ça marche même avec des émojis (presque)\n")

with open("accents.txt", "r", encoding="utf-8") as fichier:
    print(fichier.read())

In [ ]:
# Que se passe-t-il avec le mauvais encodage ?
try:
    with open("accents.txt", "r", encoding="ascii") as fichier:
        print(fichier.read())
except UnicodeDecodeError as e:
    print(f"Erreur de décodage : {e}")
    print("→ Le fichier contient des accents mais ascii ne les supporte pas")

---

## 6. Gérer les erreurs avec les fichiers

Les erreurs les plus courantes :

| Exception | Cause |
|---|---|
| `FileNotFoundError` | Le fichier n'existe pas (mode `"r"`) |
| `PermissionError` | Pas les droits de lecture/écriture |
| `IsADirectoryError` | On essaie d'ouvrir un dossier comme un fichier |
| `UnicodeDecodeError` | Mauvais encodage |

In [ ]:
# FileNotFoundError — le fichier n'existe pas
try:
    with open("fichier_inexistant.txt", "r") as fichier:
        contenu = fichier.read()
except FileNotFoundError:
    print("Erreur : ce fichier n'existe pas.")

In [ ]:
# Pattern complet et robuste pour lire un fichier
def lire_fichier(chemin):
    """Lit un fichier texte de manière sécurisée."""
    try:
        with open(chemin, "r", encoding="utf-8") as fichier:
            contenu = fichier.read()
    except FileNotFoundError:
        print(f"Erreur : '{chemin}' introuvable.")
        return None
    except PermissionError:
        print(f"Erreur : pas la permission de lire '{chemin}'.")
        return None
    except UnicodeDecodeError:
        print(f"Erreur : impossible de décoder '{chemin}' en UTF-8.")
        return None
    
    return contenu

# Test avec un fichier qui existe
resultat = lire_fichier("accents.txt")
if resultat is not None:
    print(resultat)

# Test avec un fichier qui n'existe pas
resultat = lire_fichier("nexiste_pas.txt")

---

## 7. Vérifier si un fichier existe

Le module `os` permet de vérifier l'existence d'un fichier avant de l'ouvrir.

In [ ]:
import os

# Vérifier si un fichier existe
print(f"test.txt existe ? {os.path.exists('test.txt')}")
print(f"fantome.txt existe ? {os.path.exists('fantome.txt')}")

# Vérifier si c'est un fichier ou un dossier
print(f"test.txt est un fichier ? {os.path.isfile('test.txt')}")

# Taille d'un fichier en octets
taille = os.path.getsize("test.txt")
print(f"Taille de test.txt : {taille} octets")

---

## 8. Travailler avec les chemins

### Chemins relatifs vs absolus

```python
# Relatif — par rapport au dossier courant
open("data.txt")              # dans le dossier courant
open("dossier/data.txt")      # dans un sous-dossier
open("../data.txt")           # dans le dossier parent

# Absolu — chemin complet
open("/home/user/data.txt")   # Linux/Mac
open(r"C:\Users\user\data.txt")  # Windows
```

In [ ]:
import os

# Assembler un chemin proprement (gère les / ou \ selon le système)
chemin = os.path.join("dossier", "sous_dossier", "fichier.txt")
print(f"Chemin assemblé : {chemin}")

# Extraire le nom du fichier
nom = os.path.basename("/home/user/documents/rapport.pdf")
print(f"Nom du fichier : {nom}")

# Extraire le dossier
dossier = os.path.dirname("/home/user/documents/rapport.pdf")
print(f"Dossier : {dossier}")

# Extraire l'extension
nom, extension = os.path.splitext("photo.jpg")
print(f"Nom : {nom}, Extension : {extension}")

In [ ]:
# Lister les fichiers du dossier courant
import os

fichiers = os.listdir(".")
print("Tous les fichiers :")
for f in fichiers:
    print(f"  {f}")

print()

# Filtrer seulement les .txt
fichiers_txt = [f for f in os.listdir(".") if f.endswith(".txt")]
print(f"Fichiers .txt : {fichiers_txt}")

---

## 9. Le curseur de fichier

Quand tu ouvres un fichier, Python place un **curseur** au début. Chaque lecture avance le curseur. C'est pour ça que si tu lis deux fois, la deuxième lecture retourne une chaîne vide.

In [ ]:
with open("test.txt", "r") as fichier:
    # Première lecture : le curseur part du début
    contenu1 = fichier.read()
    print(f"1ère lecture : '{contenu1.strip()}'")
    
    # Deuxième lecture : le curseur est déjà à la fin → vide !
    contenu2 = fichier.read()
    print(f"2ème lecture : '{contenu2}'  ← vide !")
    
    # seek(0) remet le curseur au début
    fichier.seek(0)
    contenu3 = fichier.read()
    print(f"3ème lecture (après seek) : '{contenu3.strip()}'")

In [ ]:
# tell() montre la position du curseur
with open("lecture_test.txt", "r") as fichier:
    print(f"Position au début : {fichier.tell()}")
    
    fichier.read(10)   # lit 10 caractères
    print(f"Après 10 caractères : {fichier.tell()}")
    
    fichier.readline()  # lit le reste de la ligne
    print(f"Après readline : {fichier.tell()}")

---

## 10. Cas pratiques

### 10.1 — Lire un fichier CSV à la main

In [ ]:
# Création d'un fichier CSV de test
with open("eleves.csv", "w", encoding="utf-8") as fichier:
    fichier.write("nom,age,note\n")
    fichier.write("Alice,20,15\n")
    fichier.write("Bob,22,12\n")
    fichier.write("Charlie,21,18\n")
    fichier.write("ligne_invalide\n")    # ligne volontairement incorrecte

print("Fichier CSV créé.")

In [ ]:
# Lecture du fichier CSV
eleves = []

with open("eleves.csv", "r", encoding="utf-8") as fichier:
    # On saute la première ligne (en-têtes)
    en_tete = fichier.readline()
    
    for numero, ligne in enumerate(fichier, 2):  # 2 car on a déjà lu la ligne 1
        ligne = ligne.strip()
        
        # On ignore les lignes vides
        if ligne == "":
            continue
        
        # On découpe par la virgule
        try:
            nom, age, note = ligne.split(",")
            eleves.append({
                "nom": nom,
                "age": int(age),
                "note": float(note)
            })
        except ValueError:
            print(f"Ligne {numero} ignorée (format invalide) : {ligne}")

# Résultat
print(f"\n{len(eleves)} élève(s) chargé(s) :")
for eleve in eleves:
    print(f"  {eleve['nom']} - {eleve['age']} ans - {eleve['note']}/20")

### 10.2 — Écrire un fichier CSV

In [ ]:
# On réutilise la liste d'élèves
eleves = [
    {"nom": "Alice", "age": 20, "note": 15},
    {"nom": "Bob", "age": 22, "note": 12},
    {"nom": "Charlie", "age": 21, "note": 18}
]

with open("export.csv", "w", encoding="utf-8") as fichier:
    # On écrit l'en-tête
    fichier.write("nom,age,note\n")
    
    # On écrit chaque élève
    for eleve in eleves:
        fichier.write(f"{eleve['nom']},{eleve['age']},{eleve['note']}\n")

# Vérification
with open("export.csv", "r") as fichier:
    print(fichier.read())

### 10.3 — Chercher un mot dans un fichier

In [ ]:
# Création d'un fichier de test
with open("article.txt", "w", encoding="utf-8") as fichier:
    fichier.write("Python est un langage de programmation.\n")
    fichier.write("Il est utilisé en data science.\n")
    fichier.write("Python est aussi utilisé pour le web.\n")
    fichier.write("Java est un autre langage populaire.\n")
    fichier.write("Python reste le plus facile à apprendre.\n")


def chercher(chemin, mot):
    """Cherche un mot dans un fichier et affiche les lignes correspondantes."""
    try:
        with open(chemin, "r", encoding="utf-8") as fichier:
            resultats = []
            for numero, ligne in enumerate(fichier, 1):
                # Recherche insensible à la casse
                if mot.lower() in ligne.lower():
                    resultats.append((numero, ligne.strip()))
    except FileNotFoundError:
        print(f"Erreur : '{chemin}' introuvable.")
        return
    
    if len(resultats) == 0:
        print(f"'{mot}' non trouvé.")
    else:
        print(f"'{mot}' trouvé {len(resultats)} fois :")
        for numero, ligne in resultats:
            print(f"  Ligne {numero} : {ligne}")


chercher("article.txt", "python")
print()
chercher("article.txt", "java")
print()
chercher("article.txt", "ruby")

### 10.4 — Remplacer du texte dans un fichier

In [ ]:
def remplacer(chemin, ancien, nouveau):
    """Remplace toutes les occurrences d'un texte dans un fichier."""
    # Étape 1 : lire le contenu
    try:
        with open(chemin, "r", encoding="utf-8") as fichier:
            contenu = fichier.read()
    except FileNotFoundError:
        print(f"Erreur : '{chemin}' introuvable.")
        return
    
    # Étape 2 : compter les occurrences
    nb = contenu.count(ancien)
    if nb == 0:
        print(f"'{ancien}' non trouvé dans le fichier.")
        return
    
    # Étape 3 : remplacer et réécrire
    nouveau_contenu = contenu.replace(ancien, nouveau)
    with open(chemin, "w", encoding="utf-8") as fichier:
        fichier.write(nouveau_contenu)
    
    print(f"{nb} occurrence(s) de '{ancien}' remplacée(s) par '{nouveau}'.")


# Avant
print("AVANT :")
with open("article.txt", "r") as f:
    print(f.read())

# Remplacement
remplacer("article.txt", "Python", "Python 3")

# Après
print("\nAPRÈS :")
with open("article.txt", "r") as f:
    print(f.read())

### 10.5 — Fichier de configuration (clé=valeur)

In [ ]:
# Création d'un fichier de config
with open("config.txt", "w") as fichier:
    fichier.write("# Ceci est un commentaire\n")
    fichier.write("\n")
    fichier.write("nom=Alice\n")
    fichier.write("theme=sombre\n")
    fichier.write("langue=fr\n")
    fichier.write("taille_police=14\n")


def lire_config(chemin):
    """Lit un fichier clé=valeur et retourne un dictionnaire."""
    config = {}
    
    try:
        with open(chemin, "r", encoding="utf-8") as fichier:
            for numero, ligne in enumerate(fichier, 1):
                ligne = ligne.strip()
                
                # On ignore les lignes vides et les commentaires
                if ligne == "" or ligne.startswith("#"):
                    continue
                
                # On découpe par le =
                try:
                    cle, valeur = ligne.split("=", 1)  # split max 1 fois
                    config[cle.strip()] = valeur.strip()
                except ValueError:
                    print(f"Ligne {numero} ignorée : {ligne}")
    
    except FileNotFoundError:
        print(f"'{chemin}' introuvable, config vide.")
    
    return config


def ecrire_config(chemin, config):
    """Écrit un dictionnaire dans un fichier de configuration."""
    with open(chemin, "w", encoding="utf-8") as fichier:
        fichier.write("# Fichier de configuration\n\n")
        for cle, valeur in config.items():
            fichier.write(f"{cle}={valeur}\n")
    print(f"Configuration sauvegardée.")


# Lire la config
config = lire_config("config.txt")
print(f"Config chargée : {config}")
print(f"Thème actuel : {config['theme']}")

# Modifier et sauvegarder
config["theme"] = "clair"
ecrire_config("config.txt", config)

# Vérification
print("\nContenu du fichier :")
with open("config.txt", "r") as f:
    print(f.read())

---

## 11. Fiche mémo

```python
# === LIRE ===
with open("fichier.txt", "r", encoding="utf-8") as f:
    tout     = f.read()            # tout d'un coup → string
    ligne    = f.readline()        # une ligne → string
    lignes   = f.readlines()       # toutes les lignes → liste
    for ligne in f:                # ligne par ligne (recommandé)
        print(ligne.strip())

# === ÉCRIRE (écrase) ===
with open("fichier.txt", "w", encoding="utf-8") as f:
    f.write("texte\n")            # une string (sans \n auto)
    f.writelines(liste)            # une liste (sans \n auto)
    print("texte", file=f)         # avec \n automatique

# === AJOUTER (à la fin) ===
with open("fichier.txt", "a", encoding="utf-8") as f:
    f.write("nouvelle ligne\n")

# === VÉRIFIER ===
import os
os.path.exists("fichier.txt")      # True/False
os.path.isfile("fichier.txt")      # est-ce un fichier ?
os.path.getsize("fichier.txt")     # taille en octets

# === ERREURS ===
# FileNotFoundError  → fichier introuvable
# PermissionError    → pas les droits
# UnicodeDecodeError → mauvais encodage
```

### Les règles d'or

1. **Toujours utiliser `with`** → le fichier se ferme automatiquement
2. **Toujours préciser `encoding="utf-8"`** → évite les problèmes d'accents
3. **Utiliser `for ligne in fichier`** → économe en mémoire
4. **Ne pas oublier les `\n`** → `write()` ne les ajoute pas
5. **`"w"` écrase tout** → utilise `"a"` pour ajouter
6. **Toujours gérer les erreurs** → `FileNotFoundError` au minimum
7. **`.strip()` sur chaque ligne** → enlève les `\n` et espaces

---

## Nettoyage des fichiers de test

Exécute cette cellule pour supprimer tous les fichiers créés pendant le tutoriel.

In [ ]:
import os

fichiers_test = [
    "test.txt", "demo_write.txt", "demo_writelines.txt",
    "demo_print.txt", "log.txt", "ecrase.txt", "ajoute.txt",
    "lecture_test.txt", "accents.txt", "eleves.csv", "export.csv",
    "article.txt", "config.txt"
]

for fichier in fichiers_test:
    if os.path.exists(fichier):
        os.remove(fichier)
        print(f"Supprimé : {fichier}")

print("\nNettoyage terminé.")